### IMPORTS AND LOAD SAVED DATA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split 
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import os 

hourly_power = pd.read_csv("../results/hourly_power.csv", index_col=0, parse_dates=True).squeeze()
anomalies = pd.read_csv("../results/anomalies.csv", index_col=0, parse_dates=True)

print(anomalies.shape)
print(anomalies['anomaly_type'].value_counts())


### FEATURE ENGINEERING

In [ ]:
def build_features(series):
    """
    Build time-series features for each data point.
    These capture local context that helps distinguish anomaly types
    """
    df = pd.DataFrame({'Value': series})

    # Rolling statistics
    for w in [3, 6, 24]:
        df[f"rolling_mean_{w}h"] = series.rolling(w, min_periods=1).mean()
        df[f"rolling_std_{w}h"] = series.rolling(w, min_periods=1).std().fillna(0)
        df[f"rolling_var_{w}h"] = series.rolling(w, min_periods=1).var().fillna(0)

    # Deviation from local mean
    df['dev_from_mean_6h'] = (series - df['rolling_mean_6h']).abs()
    df['dev_from_mean_24h'] = (series - df['rolling_mean_24h']).abs()

    # Lag features (what happened before/after)
    df['lag_1h'] = series.shift(1)
    df['lag_2h'] = series.shift(2)
    df['lead_1h'] = series.shift(-1)

    # Time of day features
    df['hour'] = series.index.hour
    df['day_of_week'] = series.index.dayofweek
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

    # Fill Nans 
    df = df.fillna(df.median())
    return df.drop(columns=['Value'])

features = build_features(hourly_power)
print(f"Feature matrix shape: {features.shape}")
print(features.columns.tolist())


### PREPARE TRAINING DATA

In [ ]:
# Tomorrow: Implement the rest of cells, understand intuition for choice of features, how binning was done since data is time-series. 